# Lesson 6.3 — Route 53: Routing Policies, Health Checks & DNS Firewall

**Level: Intermediate** | **Exam:** SOA-C03 Domain 5, Task 5.2; SAP-C02 Domain 2, Task 2.4

## Learning Objectives
- Configure Route 53 hosted zones and record types (A, AAAA, CNAME, ALIAS)
- Implement routing policies: Failover, Weighted, Latency, Geolocation
- Set up Route 53 health checks with SNS alerting
- Configure Route 53 Resolver for hybrid DNS (on-premises ↔ AWS)
- Use Route 53 DNS Firewall for malware domain protection

_Reference: [Route 53 Docs](https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/) | [Route 53 Resolver](https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/resolver.html)_

## DNS Record Types

| Type | What it does | Example |
|------|-------------|--------|
| **A** | Domain → IPv4 | `api.example.com → 1.2.3.4` |
| **AAAA** | Domain → IPv6 | `api.example.com → 2001:db8::1` |
| **CNAME** | Alias one domain to another | `www → myapp.us-east-1.elb.amazonaws.com` |
| **ALIAS** | AWS-specific: point to AWS resources (no CNAME limitation) | `example.com → alb.amazonaws.com` |
| **MX** | Mail exchange | `example.com → mail.example.com` |
| **TXT** | Text (SPF, domain verification) | `_dmarc.example.com → ...` |
| **NS** | Name servers for a zone | Delegation |

**ALIAS vs CNAME:**
- CNAME cannot be used for the zone apex (e.g., `example.com` — not `www.example.com`)
- ALIAS can point to ALB, CloudFront, S3 website, API Gateway at the zone apex
- ALIAS queries are free; CNAME queries are charged

## Route 53 Routing Policies

| Policy | Algorithm | Use Case |
|--------|-----------|----------|
| **Simple** | Single record | Static website, single endpoint |
| **Weighted** | % distribution | A/B testing (90/10), migration traffic shifting |
| **Latency** | Lowest network latency | Global app with multi-region deployment |
| **Failover** | Primary/Secondary with health check | DR (active/passive) |
| **Geolocation** | User's country/continent | Regional content, legal compliance |
| **Geoproximity** | Geographic distance with bias | Fine control over traffic distribution |
| **Multivalue** | Up to 8 healthy IPs | Simple client-side load balancing |
| **IP-based** | Based on CIDR range | Route office traffic to different endpoint |

## Route 53 Resolver

Enables DNS resolution between AWS VPC and on-premises networks:
```
On-premises server → DNS query → Resolver Inbound Endpoint → Route 53
EC2 instance → DNS query for on-prem domain → Resolver Outbound Endpoint → On-prem DNS
```

Use **Resolver Rules** (forwarding rules) to direct queries for specific domains to on-prem DNS resolvers.

In [ ]:
import boto3, json

route53 = boto3.client('route53')
route53resolver = boto3.client('route53resolver', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# List hosted zones
zones = route53.list_hosted_zones()['HostedZones']

if zones:
    print(f'Hosted Zones ({len(zones)}):')
    for z in zones:
        zone_type = 'PRIVATE' if z['Config']['PrivateZone'] else 'PUBLIC'
        print(f'  {z["Id"].split("/")[-1]:25} {z["Name"]:40} {zone_type} ({z["ResourceRecordSetCount"]} records)')
else:
    print('No hosted zones found.')
    print('\nCreate a hosted zone:')
    print("route53.create_hosted_zone(Name='example.com', CallerReference=str(time.time()))")
    print('\nThis costs $0.50/month per hosted zone')

In [ ]:
# Demonstrate Route 53 routing policy record structure
# (These show the API format — require an actual hosted zone to deploy)

print('Route 53 Record Examples by Routing Policy:\n')

records = [
    {
        'policy': 'Weighted (A/B Testing)',
        'record': {
            'Name': 'api.example.com',
            'Type': 'A',
            'SetIdentifier': 'v2-canary',
            'Weight': 10,         # 10% of traffic
            'AliasTarget': {'DNSName': 'v2-alb.us-east-1.elb.amazonaws.com', 'EvaluateTargetHealth': True}
        }
    },
    {
        'policy': 'Failover (DR)',
        'record': {
            'Name': 'app.example.com',
            'Type': 'A',
            'SetIdentifier': 'primary-us-east-1',
            'Failover': 'PRIMARY',
            'HealthCheckId': 'abc-123',
            'AliasTarget': {'DNSName': 'primary-alb.us-east-1.elb.amazonaws.com', 'EvaluateTargetHealth': True}
        }
    },
    {
        'policy': 'Latency (Global App)',
        'record': {
            'Name': 'app.example.com',
            'Type': 'A',
            'SetIdentifier': 'us-east-1',
            'Region': 'us-east-1',
            'AliasTarget': {'DNSName': 'us-east-alb.us-east-1.elb.amazonaws.com', 'EvaluateTargetHealth': True}
        }
    }
]

for r in records:
    print(f"Policy: {r['policy']}")
    print(f"  Record: {json.dumps(r['record'], indent=4)}")
    print()

In [ ]:
# Create a Route 53 Health Check
# Health checks cost ~$0.50/month per check

try:
    health_check = route53.create_health_check(
        CallerReference='cloudops-learning-demo-hc-v1',
        HealthCheckConfig={
            'Type': 'HTTPS',
            'FullyQualifiedDomainName': 'httpbin.org',
            'Port': 443,
            'ResourcePath': '/status/200',
            'RequestInterval': 30,     # Check every 30 seconds
            'FailureThreshold': 3,     # Fail after 3 consecutive failures
            'EnableSNI': True,
            'Regions': ['us-east-1', 'us-west-2', 'eu-west-1']  # Health checkers in these regions
        }
    )
    hc_id = health_check['HealthCheck']['Id']
    print(f'Created health check: {hc_id}')
    print(f'Monitoring: https://httpbin.org/status/200 from 3 regions')
except route53.exceptions.HealthCheckAlreadyExists:
    hcs = route53.list_health_checks()['HealthChecks']
    hc_id = hcs[0]['Id'] if hcs else None
    print(f'Health check exists: {hc_id}')

# List health checks
hcs = route53.list_health_checks()['HealthChecks']
if hcs:
    print(f'\nHealth Checks ({len(hcs)}):')
    for hc in hcs[:5]:
        config = hc['HealthCheckConfig']
        fqdn = config.get('FullyQualifiedDomainName', 'N/A')
        hc_type = config.get('Type', 'N/A')
        print(f'  {hc["Id"]:40} {hc_type:10} {fqdn}')

## Exercise

In [ ]:
# TODO: For each DNS requirement below, choose the correct Route 53 routing policy
# and write a brief justification

requirements = [
    {
        'requirement': 'Route 30% of users to a new version of API, 70% to old version',
        'policy': 'TODO',
        'reason': 'TODO'
    },
    {
        'requirement': 'Users in Germany must be served from EU datacenter (GDPR compliance)',
        'policy': 'TODO',
        'reason': 'TODO'
    },
    {
        'requirement': 'Global SaaS app — each user gets fastest response time regardless of location',
        'policy': 'TODO',
        'reason': 'TODO'
    },
    {
        'requirement': 'Primary region goes down → automatically serve from secondary region',
        'policy': 'TODO',
        'reason': 'TODO'
    },
]

for r in requirements:
    print(f"Requirement: {r['requirement']}")
    print(f"  Policy: {r['policy']}")
    print(f"  Reason: {r['reason']}")
    print()

In [ ]:
# Cleanup health checks created in this lesson
import boto3
route53 = boto3.client('route53')
hcs = route53.list_health_checks()['HealthChecks']
for hc in hcs:
    if hc['HealthCheckConfig'].get('FullyQualifiedDomainName') == 'httpbin.org':
        route53.delete_health_check(HealthCheckId=hc['Id'])
        print(f'Deleted health check: {hc["Id"]}')

## Summary
- **ALIAS records** are free, work at zone apex, and automatically resolve to AWS resource IPs
- **Weighted**: Traffic splitting and A/B testing. **Failover**: DR active/passive. **Latency**: Best performance.
- Health checks cost $0.50/month — always attach to Failover records
- Route 53 Resolver **inbound endpoint** = on-prem can resolve AWS private hostnames
- Route 53 Resolver **outbound endpoint** = EC2 can resolve on-prem domain names

---
**Next:** [Lesson 6.4 — CloudFront, WAF, Shield & Global Accelerator](lesson-6.4-cloudfront-waf.ipynb)